In [0]:
# Install required library to read Excel files
%pip install openpyxl

In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# AUDIENCE PROFILING
# Using df_memb for customer level analysis and rfm table which already has segments
# -----------------------------------------------------------------------------------

# Load cleaned member data
df_memb = pd.read_parquet("/Volumes/workspace/default/retail_data/df_memb.parquet")

# Load RFM table with segments already assigned
rfm = pd.read_parquet("/Volumes/workspace/default/retail_data/rfm.parquet")

# Merge transaction data with segments
# This gives us every transaction labelled with its customer's segment
df_profiling = df_memb.merge(rfm[['Customer ID', 'Segment']], on='Customer ID', how='left')

print(f"Data loaded successfully")
print(f"\nMember transactions: {len(df_memb):,}")
print(f"Customers with segments: {len(rfm):,}")
print(f"Profiling dataset: {len(df_profiling):,} rows")
print(f"\nSegments in dataset:")
print(df_profiling['Segment'].value_counts())

print(f"\nLoad top 10 rows")
print(df_profiling.head(10).to_string(index=False))

In [0]:
# GEOGRAPHIC PROFILING BY SEGMENT
# Which countries do each segment come from?
# Helps identify where to focus retention and acquisition efforts geographically
# -------------------------------------------------------------------------------

# Get unique customer per segment and country
# We use rfm merged with df_memb to get country per customer
customer_country = df_memb.groupby('Customer ID')['Country'].first().reset_index()

# Merge with RFM to get segment + country per customer
segment_country = rfm[['Customer ID', 'Segment']].merge(customer_country, on='Customer ID')

# Count customers per segment per country
geo_profile = segment_country.groupby(['Segment', 'Country'])['Customer ID'].count().reset_index()
geo_profile.columns = ['Segment', 'Country', 'Customer Count']

# Focus on top 5 countries per segment
top_countries_per_segment = geo_profile.sort_values(
    ['Segment', 'Customer Count'], ascending=[True, False]
).groupby('Segment').head(5)

# Plot
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

segments = rfm['Segment'].unique()

for i, segment in enumerate(segments):
    data = top_countries_per_segment[top_countries_per_segment['Segment'] == segment]
    bars = axes[i].barh(data['Country'], data['Customer Count'], color='#4C72B0')
    axes[i].barh(data['Country'], data['Customer Count'], color='#4C72B0')
    axes[i].set_title(f'{segment}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Customers', fontsize=9)
    axes[i].invert_yaxis()

    # Add labels to each bar
    for bar in bars:
        axes[i].text(
            bar.get_width() + 1,
            bar.get_y() + bar.get_height() / 2,
            f'{int(bar.get_width())}',
            va='center',
            fontsize=12
        )

plt.suptitle('Top 5 Countries per Customer Segment', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [0]:
# PRODUCT PRICE TIER CLASSIFICATION
# Categorising all products into price bands
# Budget    → under £2
# Mid Range → £2 to £5
# Premium   → £5 to £15
# Luxury    → above £15
# ============================================

# Get unique products with their average unit price
product_prices = df_profiling.groupby(
    'Description')['Price'].mean().reset_index()
product_prices.columns = ['Product', 'Avg_Price']

# Classify products into price tiers
def classify_price_tier(price):
    if price < 2:
        return 'Budget (under £2)'
    elif price < 5:
        return 'Mid Range (£2-£5)'
    elif price < 15:
        return 'Premium (£5-£15)'
    else:
        return 'Luxury (above £15)'

product_prices['Price_Tier'] = product_prices['Avg_Price'].apply(classify_price_tier)

# Count products per tier
tier_counts = product_prices['Price_Tier'].value_counts().reset_index()
tier_counts.columns = ['Price_Tier', 'Product_Count']

# Calculate percentage
tier_counts['Percentage'] = (
    tier_counts['Product_Count'] / tier_counts['Product_Count'].sum() * 100
).round(2)

print("PRODUCT PRICE TIER BREAKDOWN")
print(tier_counts.to_string(index=False))

# Plot
plt.figure(figsize=(10, 6))
colors = ['#e74c3c', '#f39c12', '#2ecc71', '#4C72B0']
bars = plt.barh(
    tier_counts['Price_Tier'],
    tier_counts['Product_Count'],
    color=colors
)

# Add labels
for bar, pct in zip(bars, tier_counts['Percentage']):
    plt.text(
        bar.get_width() + 10,
        bar.get_y() + bar.get_height() / 2,
        f'{int(bar.get_width())} products ({pct}%)',
        va='center',
        fontsize=10
    )

plt.title('Product Catalogue by Price Tier', fontsize=16, fontweight='bold')
plt.xlabel('Number of Products', fontsize=12)
plt.tight_layout()
plt.show()

# Show top 5 products per tier
print("\n - TOP 5 PRODUCTS (LUXURY) -")
print(product_prices[product_prices['Price_Tier'] == 'Luxury (above £15)'].sort_values(
    'Avg_Price', ascending=False).head(5).to_string(index=False))

print("\n - TOP 5 PRODUCTS (PREMIUM) -")
print(product_prices[product_prices['Price_Tier'] == 'Premium (£5-£15)'].sort_values(
    'Avg_Price', ascending=True).head(5).to_string(index=False))

print("\n - TOP 5 PRODUCTS (MID RANGE) -")
print(product_prices[product_prices['Price_Tier'] == 'Mid Range (£2-£5)'].sort_values(
    'Avg_Price', ascending=True).head(5).to_string(index=False))

print("\n - TOP 5 PRODUCTS (BUDGET) -")
print(product_prices[product_prices['Price_Tier'] == 'Budget (under £2)'].sort_values(
    'Avg_Price', ascending=True).head(5).to_string(index=False))

In [0]:
# BUYING BEHAVIOUR ANALYSIS
# Are customers bulk buyers or premium buyers?
# Three metrics tell the full story:
# 1. Average unit price — cheap or expensive products?
# 2. Average quantity per order — bulk or individual?
# 3. Average order value — total spend per order
# -----------------------------------------------------

# Calculate average unit price per customer
avg_unit_price = df_profiling.groupby(
    ['Customer ID', 'Segment'])['Price'].mean().reset_index()
avg_unit_price.columns = ['Customer ID', 'Segment', 'Avg_Unit_Price']

# Calculate average quantity per order per customer
avg_quantity = df_profiling.groupby(
    ['Customer ID', 'Segment'])['Quantity'].mean().reset_index()
avg_quantity.columns = ['Customer ID', 'Segment', 'Avg_Quantity']

# Calculate average order value per customer
avg_order_value = df_profiling.groupby(
    ['Customer ID', 'Segment'])['TotalRevenue'].mean().reset_index()
avg_order_value.columns = ['Customer ID', 'Segment', 'Avg_Order_Value']

# Merge all three together
buying_behaviour = avg_unit_price.merge(
    avg_quantity, on=['Customer ID', 'Segment']
).merge(
    avg_order_value, on=['Customer ID', 'Segment']
)

# Get segment level averages
segment_behaviour = buying_behaviour.groupby('Segment').agg(
    Avg_Unit_Price=('Avg_Unit_Price', 'mean'),
    Avg_Quantity=('Avg_Quantity', 'mean'),
    Avg_Order_Value=('Avg_Order_Value', 'mean')
).round(2).reset_index()

print("BUYING BEHAVIOUR BY SEGMENT")
print(segment_behaviour.to_string(index=False))

# Plot three charts side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

metrics = ['Avg_Unit_Price', 'Avg_Quantity', 'Avg_Order_Value']
titles = ['Avg Unit Price (£)', 'Avg Quantity per Order', 'Avg Order Value (£)']
colors = ['#4C72B0', '#2ecc71', '#e74c3c']

for i, (metric, title, color) in enumerate(zip(metrics, titles, colors)):
    # Sort by metric value for better visual
    data = segment_behaviour.sort_values(metric, ascending=True)
    bars = axes[i].barh(data['Segment'], data[metric], color=color)
    axes[i].set_title(title, fontsize=12, fontweight='bold')
    axes[i].set_xlabel(title, fontsize=10)

    # Add labels
    for bar in bars:
        axes[i].text(
            bar.get_width() + 0.1,
            bar.get_y() + bar.get_height() / 2,
            f'{bar.get_width():.2f}',
            va='center',
            fontsize=9
        )

plt.suptitle('Bulk Buyer vs Premium Buyer Analysis by Segment',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [0]:
# PRODUCT PREFERENCES BY SEGMENT
# What products does each segment buy most?
# Helps tailor marketing and product strategy per customer group
# ---------------------------------------------------------------

# Top 5 products per segment by revenue
product_segment = df_profiling.groupby(
    ['Segment', 'Description']
)['TotalRevenue'].sum().reset_index()

product_segment.columns = ['Segment', 'Product', 'Revenue']

# Get top 5 products per segment
top_products_per_segment = product_segment.sort_values(
    ['Segment', 'Revenue'], ascending=[True, False]
).groupby('Segment').head(5)

# Plot
fig, axes = plt.subplots(2, 4, figsize=(24, 12))
axes = axes.flatten()

segments = rfm['Segment'].unique()

for i, segment in enumerate(segments):
    data = top_products_per_segment[
        top_products_per_segment['Segment'] == segment
    ].sort_values('Revenue', ascending=True)
    
    bars = axes[i].barh(
        data['Product'],
        data['Revenue'],
        color='#2ecc71'
    )
    axes[i].set_title(f'{segment}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Revenue (£)', fontsize=12)
    axes[i].invert_yaxis()

    # Add labels
    for bar in bars:
        axes[i].text(
            bar.get_width() + 100,
            bar.get_y() + bar.get_height() / 2,
            f'£{int(bar.get_width()):,}',
            va='center',
            fontsize=12
        )

plt.suptitle('Top 5 Products by Revenue per Segment', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [0]:
# PURCHASE TIMING ANALYSIS
# When does each segment buy?
# Day of week and month analysis per segment
# Helps time marketing campaigns effectively
# --------------------------------------------

# Extract day of week and month from invoice date
df_profiling['DayOfWeek'] = df_profiling['InvoiceDate'].dt.day_name()
df_profiling['Month'] = df_profiling['InvoiceDate'].dt.month_name()
df_profiling['MonthNum'] = df_profiling['InvoiceDate'].dt.month

# --- PART 1: ORDERS BY DAY OF WEEK PER SEGMENT ---
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

day_segment = df_profiling.groupby(
    ['Segment', 'DayOfWeek'])['Invoice'].nunique().reset_index()
day_segment.columns = ['Segment', 'DayOfWeek', 'Order_Count']

# --- PART 2: ORDERS BY MONTH PER SEGMENT ---
month_segment = df_profiling.groupby(
    ['Segment', 'Month', 'MonthNum'])['Invoice'].nunique().reset_index()
month_segment.columns = ['Segment', 'Month', 'MonthNum', 'Order_Count']
month_segment = month_segment.sort_values('MonthNum')

# --- PLOT: DAY OF WEEK ---
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()

segments = rfm['Segment'].unique()

for i, segment in enumerate(segments):
    data = day_segment[day_segment['Segment'] == segment]
    data = data.set_index('DayOfWeek').reindex(day_order).reset_index()
    
    axes[i].bar(
        data['DayOfWeek'],
        data['Order_Count'],
        color='#4C72B0'
    )
    axes[i].set_title(f'{segment}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Day', fontsize=9)
    axes[i].set_ylabel('Orders', fontsize=9)
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Orders by Day of Week per Segment', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# --- PLOT: MONTH ---
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()

for i, segment in enumerate(segments):
    data = month_segment[month_segment['Segment'] == segment]
    
    axes[i].bar(
        data['Month'],
        data['Order_Count'],
        color='#2ecc71'
    )
    axes[i].set_title(f'{segment}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Month', fontsize=9)
    axes[i].set_ylabel('Orders', fontsize=9)
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle('Orders by Month per Segment', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

In [0]:
# SAVE PROFILING DATASET
# Saving the enriched dataset with segments, price tiers and timing columns for use in Notebook 5 — Churn Prediction
#--------------------------------------------------------------------------------------------------------------------

profiling_path = "/Volumes/workspace/default/retail_data/df_profiling.parquet"
df_profiling.to_parquet(profiling_path, index=False)

print(f"Profiling dataset saved")
print(f"\nRows: {len(df_profiling):,}")
print(f"Columns: {df_profiling.columns.tolist()}")